In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

#nltk.download('punkt_tab')
#nltk.download('stopwords')

def preprocess_text(text): #converts a string into a list of words excluding stopwords and non alphabetic
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return tokens

df = pd.read_csv('potential-talents.csv')

In [2]:
#print(df["job_title"])

0      2019 C.T. Bauer College of Business Graduate (...
1      Native English Teacher at EPIK (English Progra...
2                  Aspiring Human Resources Professional
3                 People Development Coordinator at Ryan
4        Advisory Board Member at Celal Bayar University
                             ...                        
99     Aspiring Human Resources Manager | Graduating ...
100                Human Resources Generalist at Loparex
101     Business Intelligence and Analytics at Travelers
102                       Always set them up for Success
103     Director Of Administration at Excellence Logging
Name: job_title, Length: 104, dtype: str


In [ ]:
def phrase_vector(text, model):
    tokens = preprocess_text(text)

    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if not vectors:
        return None

    return np.mean(vectors, axis=0)

In [3]:
preprocessed_sentences = [preprocess_text(sentence) for sentence in df['job_title']] 
#performing defined function on each element

model = Word2Vec(sentences=preprocessed_sentences, vector_size=100, window=5, min_count=1, workers=4, sg=1)

model.save("word2vec_ranking.model") #saves to binary file

In [ ]:
targets = [
    "aspiring human resources",
    "seeking human resources"
]

target_vectors = {
    target: phrase_vector(target, model)
    for target in targets
}

In [ ]:
candidate_vectors = df["job_title"].apply(
    lambda title: phrase_vector(str(title), model)
)

for target, target_vec in target_vectors.items():
    scores = candidate_vectors.apply(
        lambda vec: (
            float(cosine_similarity(
                target_vec.reshape(1, -1),
                vec.reshape(1, -1)
            )[0, 0])
            if vec is not None and target_vec is not None
            else np.nan
        )
    )

    df[f"similarity_{target.replace(' ', '_')}"] = scores

In [ ]:
df.sort_values(
    "similarity_aspiring_human_resources",
    ascending=False
)[["job_title", "similarity_aspiring_human_resources"]].head(10)

In [ ]:
df.sort_values(
    "similarity_seeking_human_resources",
    ascending=False
)[["job_title", "similarity_seeking_human_resources"]].head(10)

In [4]:
#assert False

In [5]:
#print(preprocessed_sentences)

In [8]:
#print(words)

['human', 'resources', 'aspiring', 'professional', 'student', 'seeking', 'generalist', 'college', 'specialist', 'university', 'business', 'coordinator', 'english', 'manager', 'humber', 'laude', 'cum', 'magna', 'graduate', 'bauer', 'senior', 'hr', 'ryan', 'development', 'people', 'management', 'korea', 'program', 'epik', 'teacher', 'native', 'opportunities', 'atlanta', 'buckhead', 'intercontinental', 'sphr', 'gphr', 'energy', 'woodlands', 'houston', 'engie', 'officer', 'csr', 'communications', 'marketing', 'chro', 'svp', 'chapman', 'positions', 'hris', 'bayar', 'celal', 'member', 'board', 'advisory', 'major', 'position', 'director', 'internship', 'leader', 'systems', 'information', 'america', 'north', 'staffing', 'retail', 'logging', 'excellence', 'administration', 'success', 'set', 'always', 'travelers', 'analytics', 'intelligence', 'loparex', 'louis', 'may', 'graduating', 'paint', 'hardware', 'delphi', 'kokomo', 'indiana', 'state', 'westfield', 'relocation', 'travel', 'open', 'beach',